# 任务4：选定四层的自动病灶勾画

在保存完整影像、具备NVIDIA显卡的电脑上训练二维nnU-Net。每个患者拆成四张带标注的切片，三序列作为三个输入通道；同一患者四张切片始终同折。未标注层不进入训练或评价。

本阶段仍需要人工确定四层，完成的是这些层的自动勾画，不能据此声称全脑自动检出。标签1、2、3尚未说明临床含义，代码保留数字标签名。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    settings = path.parent / "experiment_settings.json"
    if settings.is_file():
        overlay = json.loads(settings.read_text(encoding="utf-8"))
        allowed = {"annotation_scope", "optimized", "radiomics", "segmentation", "report_version"}
        if set(overlay) - allowed:
            raise ValueError("experiment_settings.json 只放实验参数，机器路径仍在 config.json。")
        for key, value in overlay.items():
            if isinstance(value, dict):
                cfg[key] = {**cfg.get(key, {}), **value}
            else:
                cfg[key] = value
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg




def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics" / experiment_name(cfg, "radiomics", "selected2d_v1"),
        "explain": root / "explain", "segmentation": root / "segmentation" / experiment_name(cfg, "segmentation", "selected2d_v1"),
        "optimized": root / "optimized" / experiment_name(cfg, "optimized", "resnet_v1"),
        "reports": root / "reports" / safe_name(cfg.get("report_version", "comparison_v2")),
    }




def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")




def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))




def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()




def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result




def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "resnet_baseline_and_optimized_selected_slices", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)




def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table




def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)




def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)




def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    manifest["artifacts"] = {portable_name(k): v for k, v in manifest["artifacts"].items()}
    for key in ("cache", "cohort"):
        manifest[key] = portable_name(manifest[key])
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")


    # Notebook hashes are provenance, not data identity. Model contracts are checked separately.


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本操作需要原分辨率影像，请在数据电脑执行；导入结果和生成报告无需原影像。")




def safe_name(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("实验名称只使用字母、数字、下划线或短横线。")
    return value




def experiment_name(cfg, stage, default):
    return safe_name(cfg.get(stage, {}).get("experiment", default))




def portable_name(value):
    from pathlib import PurePosixPath
    name = str(value).replace("\\", "/")
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or any(p in ("..", ".") or ":" in p for p in path.parts):
        raise ValueError(f"数据包必须使用包内相对路径：{value}")
    return path.as_posix()




def contained_file(root, name):
    root = Path(root).resolve()
    target = (root / portable_name(name)).resolve()
    target.relative_to(root)
    return target




def cohort_identity(cfg):
    """Patient identity independent of local directories, CSV formatting and row order."""
    import pandas as pd
    table = read_cohort(cfg)
    columns = ["mr", "label", "split", "development_fold", "max_layer", "moca"] + CLINICAL
    if not set(columns) <= set(table):
        raise ValueError("病例清单缺少患者、结局、分组、取层或临床列。")
    rows = []
    for _, row in table.sort_values("mr").iterrows():
        values = {}
        for key in columns:
            value = row[key]
            values[key] = (None if pd.isna(value) else str(value) if key in ("mr", "split") else format(float(value), ".12g"))
        rows.append(values)
    payload = {"schema": 1, "patients": rows}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()




def definition_contract(cfg, prefix, names):
    import ast
    files = list(cfg["_root"].glob(prefix + "_*.ipynb"))
    if len(files) != 1:
        raise ValueError(f"找不到唯一的 {prefix} Notebook。")
    notebook = json.loads(files[0].read_text(encoding="utf-8"))
    definitions = {}
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
                if node.name in definitions:
                    raise ValueError(f"重复定义：{node.name}")
                definitions[node.name] = hashlib.sha256(ast.dump(node, include_attributes=False).encode()).hexdigest()
    if set(definitions) != set(names):
        raise ValueError("模型或处理流程定义不齐全。")
    return definitions




def verify_transfer(cfg, directory, kind=None):
    directory = Path(directory).resolve()
    manifest = json.loads((directory / "transfer_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema") != 1 or manifest.get("kind") not in ("radiomics_features", "radiomics_complete", "segmentation_results", "segmentation_weights"):
        raise ValueError("不支持的数据包类型或版本。")
    if kind and manifest["kind"] != kind:
        raise ValueError("数据包类型不匹配。")
    if manifest["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("两台机器的患者、标签、临床数据、取层或开发折不一致。")
    if manifest["native_source_identity"] != native_source_identity(cfg):
        raise ValueError("数据包与分类缓存记录的原始影像或人工标注不是同一版本。")
    if manifest["annotation_scope"] != "selected_slices" or cfg["annotation_scope"] != "selected_slices":
        raise ValueError("当前流程只接受四层标注协议。")
    for name, checksum in manifest["artifacts"].items():
        file = contained_file(directory, name)
        if not file.is_file() or digest(file) != checksum:
            raise ValueError(f"数据包文件缺失或传输后发生变化：{name}")
    return manifest




def export_transfer(cfg, source, filenames, destination, kind, protocol):
    import shutil
    source, destination = Path(source).resolve(), Path(destination).resolve()
    fresh(destination)
    mapping = {}
    for name in filenames:
        name = portable_name(name)
        file = contained_file(source, name)
        if not file.is_file():
            raise FileNotFoundError(file)
        mapping[name] = digest(file)
    destination.mkdir(parents=True)
    for name in mapping:
        target = contained_file(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), target)
    dump_json(destination / "transfer_manifest.json", {
        "schema": 1, "kind": kind, "cohort_identity": cohort_identity(cfg), "native_source_identity": native_source_identity(cfg),
        "annotation_scope": cfg["annotation_scope"], "protocol": protocol,
        "artifacts": mapping, "source_run_id": cfg["run_id"],
        "notebook_definitions_sha256": source_snapshot(cfg)})
    verify_transfer(cfg, destination, kind)
    return destination




def import_transfer(cfg, source, stage):
    import shutil
    source = Path(source).resolve()
    manifest = verify_transfer(cfg, source)
    allowed = {"radiomics": ("radiomics_features", "radiomics_complete"),
               "segmentation": ("segmentation_results",), "segmentation_weights": ("segmentation_weights",)}
    if stage not in allowed or manifest["kind"] not in allowed[stage]:
        raise ValueError("数据包与接收阶段不匹配。")
    expected_protocol = "selected_four_slices_2d_pixels_v1"
    if manifest["protocol"] != expected_protocol:
        raise ValueError("数据包的二维取层/像素空间处理协议不匹配。")
    target = paths(cfg)[stage] if stage != "segmentation_weights" else paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    fresh(target)
    target.mkdir(parents=True)
    for name in manifest["artifacts"]:
        dest = contained_file(target, name)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), dest)
    shutil.copy2(source / "transfer_manifest.json", target / "transfer_manifest.json")
    verify_transfer(cfg, target)
    return target




def native_source_identity(cfg):
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    patients = read_cohort(cfg).mr.tolist()
    files = IMAGE_FILES + ["Untitled.nii.gz"]
    selected = {mr: {name: audit["patients"][mr]["files"][name]["sha256"] for name in files} for mr in patients}
    return hashlib.sha256(json.dumps(selected, sort_keys=True).encode()).hexdigest()




In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 检查影像、取层和标注

复用00确定的患者、时间队列及开发折。下面只从原影像读取指定四层，并记录原空间矩阵，方便后续把预测叠回原图。


In [ ]:
"""Native inputs for the confirmed four annotated slices; no unannotated background."""
from pathlib import Path
import json
import re
import numpy as np


def native_patient_files(cfg, patient):
    require_native_images(cfg)
    columns = SEQUENCES + ["manual_mask"]
    recorded = [Path(str(patient.get(c, ""))) for c in columns]
    if all(p.is_file() for p in recorded):
        return recorded
    root = Path(cfg.get("patients_root") or "")
    if not root.is_dir():
        raise FileNotFoundError("在 config.json 填写数据电脑上的 patients_root。")
    if "_native_lookup" not in cfg:
        lookup = {}
        for file in root.rglob(IMAGE_FILES[0]):
            matches = set(re.findall(r"(?<![A-Z0-9])MR\d+(?!\d)", str(file.parent).upper()))
            for mr in matches:
                lookup.setdefault(mr, []).append(file.parent)
        cfg["_native_lookup"] = lookup
    matches = cfg["_native_lookup"].get(str(patient.mr).upper(), [])
    if len(matches) != 1:
        raise ValueError(f"{patient.mr} 未找到唯一的三序列目录；检查 patients_root 和目录中的 MR 编号。")
    files = [matches[0] / name for name in IMAGE_FILES + ["Untitled.nii.gz"]]
    if not all(file.is_file() for file in files):
        raise FileNotFoundError(f"{patient.mr} 缺少三序列影像或人工标注。")
    return files






def selected_indices(patient, depth):
    value = float(patient.max_layer)
    if not np.isfinite(value) or value != int(value):
        raise ValueError(f"{patient.mr} 起始层不是整数。")
    start = int(value) - 1
    zs = list(range(start, start + 4))
    if start < 0 or zs[-1] >= depth:
        raise ValueError(f"{patient.mr} 的四层取层越界。")
    recorded = patient.get("selected_slices_1based")
    if isinstance(recorded, str) and recorded.strip() and json.loads(recorded) != [z + 1 for z in zs]:
        raise ValueError(f"{patient.mr} 的缓存取层与标注层记录不一致。")
    return zs






def selected_patient(cfg, patient):
    import nibabel as nib
    if cfg.get("annotation_scope") != "selected_slices":
        raise ValueError("本流程要求确认全队列仅选定四层标注。")
    files = native_patient_files(cfg, patient)
    volumes = [nib.load(str(file)) for file in files]
    ref = volumes[0]
    tolerance = np.array([1e-4, 1e-4, 1e-4, 1e-3])
    if len(ref.shape) != 3 or any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, rtol=0, atol=tolerance) for v in volumes[1:]):
        raise ValueError(f"{patient.mr} 的 shape/affine 不一致，先回到00核查。")
    zs = selected_indices(patient, ref.shape[2])
    truth = np.asarray(volumes[-1].dataobj)
    if not np.isfinite(truth).all() or not set(np.unique(truth)) <= {0, 1, 2, 3}:
        raise ValueError(f"{patient.mr} 人工标签必须为0/1/2/3。")
    outside = np.ones(ref.shape[2], dtype=bool)
    outside[zs] = False
    if np.any(truth[:, :, outside] != 0):
        raise ValueError(f"{patient.mr} 在指定四层之外发现标注，请核对起始层。")
    images = [np.asarray(v.dataobj, dtype=np.float32)[:, :, zs] for v in volumes[:3]]
    if any(not np.isfinite(v).all() for v in images):
        raise ValueError(f"{patient.mr} 影像包含非有限值。")
    checksums = {file.name: digest(file) for file in files}
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    expected = audit["patients"][str(patient.mr)]["files"]
    if any(checksums[name] != expected[name]["sha256"] for name in checksums):
        raise ValueError(f"{patient.mr} 原始影像或标注与00的缓存来源不同，请先核查数据版本。")
    return images, truth[:, :, zs].astype(np.uint8), {
        "mr": str(patient.mr), "slices_1based": [z + 1 for z in zs], "shape": list(ref.shape),
        "native_affine": ref.affine.tolist(), "working_space": "pixel",
        "files_sha256": checksums}






## 数据集、训练和结果转移工具

每层保存成一个单层NIfTI，平面分辨率保持原样，工作网格采用像素单位。分割模型训练使用开发患者的标注，时间验证患者只导出影像。自定义划分在预处理后写入，防止同一患者不同层跨折。

`evaluate_masks` 逐患者、逐标签计算四层合并Dice；HD95在各层平面内计算，单位是像素。两侧都空的病例不计入Dice均值，单侧为空保留为失败；某层单侧为空时，患者该标签的HD95记为未定义。

格式和划分参考 [nnU-Net数据格式](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/dataset_format.md)及[自定义划分](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/manual_data_splits.md)。


In [ ]:
"""Patient-grouped 2D nnU-Net on the four annotated slices, with portable artifacts."""
from __future__ import annotations
import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd


def locations(cfg):
    root = paths(cfg)["segmentation"]
    dataset_id = int(cfg["segmentation"]["dataset_id"])
    if not 100 <= dataset_id <= 999 or cfg["segmentation"]["configuration"] != "2d":
        raise ValueError("四层标注使用100～999的数据集ID和2d配置。")
    name = f"Dataset{dataset_id:03d}_PSCISelected2D"
    return {"root": root, "name": name, "raw": root / "nnUNet_raw" / name,
        "preprocessed": root / "nnUNet_preprocessed" / name, "results": root / "nnUNet_results",
        "validation": root / "validation_predictions", "oof": root / "development_oof_predictions"}






def segmentation_label_names(cfg):
    # Numeric placeholders preserve IDs without inventing a clinical interpretation.
    names = {str(i): cfg.get("label_names", {}).get(str(i)) or f"label_{i}" for i in (1, 2, 3)}
    if len(set(names.values()) | {"background"}) != 4:
        raise ValueError("标签名称须唯一且不能命名为background。")
    return names






def export_dataset(cfg):
    import nibabel as nib
    require_native_images(cfg)
    loc, cohort = locations(cfg), read_cohort(cfg)
    fresh(loc["root"])
    names = segmentation_label_names(cfg)
    for directory in ("imagesTr", "labelsTr", "imagesTs"):
        (loc["raw"] / directory).mkdir(parents=True, exist_ok=True)
    rows, sources = [], []
    # Artificial working geometry: one slice, original in-plane resolution, distances in pixels.
    affine = np.diag([1.0, 1.0, 999.0, 1.0])
    for _, patient in cohort.iterrows():
        images, truth, source = selected_patient(cfg, patient)
        sources.append(source)
        training = patient.split == "development"
        folder = loc["raw"] / ("imagesTr" if training else "imagesTs")
        for index, layer in enumerate(source["slices_1based"]):
            case = f"{patient.mr}_s{layer:03d}"
            rows.append({"case_id": case, "mr": patient.mr, "slice_1based": layer,
                         "slice_offset": index, "split": patient.split, "development_fold": int(patient.development_fold)})
            for channel, volume in enumerate(images):
                nib.save(nib.Nifti1Image(volume[:, :, index:index + 1].astype(np.float32), affine), folder / f"{case}_{channel:04d}.nii.gz")
            if training:
                nib.save(nib.Nifti1Image(truth[:, :, index:index + 1], affine), loc["raw"] / "labelsTr" / f"{case}.nii.gz")
    cases = pd.DataFrame(rows)
    cases.to_csv(loc["root"] / "case_index.csv", index=False)
    dev = cases.query("split == 'development'")
    if set(dev.development_fold) != {1, 2, 3, 4, 5}:
        raise ValueError("开发患者必须沿用00中的五折。")
    splits = []
    for fold in range(1, 6):
        valid = dev.loc[dev.development_fold == fold, "case_id"].tolist()
        train = dev.loc[dev.development_fold != fold, "case_id"].tolist()
        splits.append({"train": train, "val": valid})
        inputs = loc["root"] / "oof_inputs" / f"fold_{fold - 1}"
        inputs.mkdir(parents=True)
        for case in valid:
            for channel in range(3):
                name = f"{case}_{channel:04d}.nii.gz"
                shutil.copy2(loc["raw"] / "imagesTr" / name, inputs / name)
    dump_json(loc["raw"] / "dataset.json", {"channel_names": {"0": "T1-FLAIR", "1": "T2", "2": "DWI"},
        "labels": {"background": 0, **{name: int(label) for label, name in names.items()}},
        "numTraining": len(dev), "file_ending": ".nii.gz"})
    dump_json(loc["root"] / "splits_final.json", splits)
    tracked = [file for directory in (loc["raw"], loc["root"] / "oof_inputs") for file in directory.rglob("*") if file.is_file()]
    tracked += [loc["root"] / "case_index.csv", loc["root"] / "splits_final.json"]
    dump_json(loc["root"] / "dataset_provenance.json", {"cohort_identity": cohort_identity(cfg),
        "validation_labels_in_training": False, "annotation_scope": "selected_slices", "configuration": "2d",
        "protocol": "selected_four_slices_2d_pixels_v1", "label_names": names, "sources": sources,
        "artifacts": {file.relative_to(loc["root"]).as_posix(): digest(file) for file in tracked}})
    record(cfg, "segmentation_dataset", {"protocol": "selected_four_slices_2d_pixels_v1"})
    return loc






def verify_dataset(cfg):
    loc, cohort = locations(cfg), read_cohort(cfg)
    meta = json.loads((loc["root"] / "dataset_provenance.json").read_text(encoding="utf-8"))
    if meta["cohort_identity"] != cohort_identity(cfg) or meta["annotation_scope"] != cfg["annotation_scope"]:
        raise ValueError("分割导出后患者、划分、取层或标注协议发生变化。")
    if meta["configuration"] != cfg["segmentation"]["configuration"] or meta["label_names"] != segmentation_label_names(cfg):
        raise ValueError("分割配置或标签定义发生变化，请更换 segmentation.experiment。")
    for name, checksum in meta["artifacts"].items():
        if digest(loc["root"] / name) != checksum:
            raise ValueError(f"分割数据文件变化：{name}")
    dev = cohort.query("split == 'development'")
    expected = {f"{patient.mr}_s{layer:03d}" for _, patient in dev.iterrows()
                for layer in range(int(patient.max_layer), int(patient.max_layer) + 4)}
    actual = {file.name.removesuffix(".nii.gz") for file in (loc["raw"] / "labelsTr").glob("*.nii.gz")}
    if actual != expected:
        raise ValueError("labelsTr与开发患者四层清单不一致。")






def command_plan(cfg):
    loc, settings = locations(cfg), cfg["segmentation"]
    dataset = str(settings["dataset_id"])
    plan = {"preprocess": [["nnUNetv2_plan_and_preprocess", "-d", dataset, "-c", "2d", "--verify_dataset_integrity"]],
            "train": [], "predict_oof": [], "predict_validation": []}
    for fold in range(5):
        plan["train"].append(["nnUNetv2_train", dataset, "2d", str(fold), "--npz", "-device", settings["device"]])
        plan["predict_oof"].append(["nnUNetv2_predict", "-i", str(loc["root"] / "oof_inputs" / f"fold_{fold}"),
            "-o", str(loc["oof"] / f"fold_{fold}"), "-d", dataset, "-c", "2d", "-f", str(fold), "-device", settings["device"]])
    plan["predict_validation"] = [["nnUNetv2_predict", "-i", str(loc["raw"] / "imagesTs"),
        "-o", str(loc["validation"]), "-d", dataset, "-c", "2d", "-f", "0", "1", "2", "3", "4", "-device", settings["device"]]]
    return plan






def install_splits(cfg):
    verify_dataset(cfg)
    loc = locations(cfg)
    if not loc["preprocessed"].is_dir():
        raise FileNotFoundError("先完成预处理，再安装固定的患者划分。")
    source, target = loc["root"] / "splits_final.json", loc["preprocessed"] / "splits_final.json"
    if target.exists():
        if json.loads(target.read_text()) != json.loads(source.read_text()):
            raise ValueError("已有nnU-Net划分与固定患者折不同，不覆盖。")
    else:
        shutil.copy2(source, target)
    return target






def run_phase(cfg, phase):
    verify_dataset(cfg)
    loc = locations(cfg)
    if phase not in command_plan(cfg):
        raise ValueError("未知分割阶段。")
    if phase != "preprocess":
        install_splits(cfg)
    env = os.environ.copy()
    env.update(nnUNet_raw=str(loc["raw"].parent), nnUNet_preprocessed=str(loc["preprocessed"].parent), nnUNet_results=str(loc["results"]))
    for folder in (loc["preprocessed"].parent, loc["results"]):
        folder.mkdir(parents=True, exist_ok=True)
    for command in command_plan(cfg)[phase]:
        print(shlex.join(command), flush=True)
        subprocess.run(command, check=True, env=env)
    record(cfg, "segmentation_" + phase, {"protocol": "selected_four_slices_2d_pixels_v1"})






def predicted_slices(cfg, patient, scope, shape):
    import nibabel as nib
    loc = locations(cfg)
    directory = loc["validation"] if scope == "validation" else loc["oof"] / f"fold_{int(patient.development_fold) - 1}"
    arrays, checksums = [], {}
    for layer in range(int(patient.max_layer), int(patient.max_layer) + 4):
        name = f"{patient.mr}_s{layer:03d}.nii.gz"
        volume = nib.load(str(directory / name))
        values = np.asarray(volume.dataobj)
        if values.shape != (*shape, 1) or not np.allclose(volume.affine, np.diag([1., 1., 999., 1.]), rtol=0, atol=1e-4):
            raise ValueError(f"{name} 预测工作网格不一致。")
        if not np.isfinite(values).all() or not set(np.unique(values)) <= {0, 1, 2, 3}:
            raise ValueError(f"{name} 预测标签不是0/1/2/3。")
        arrays.append(values[:, :, 0].astype(np.uint8))
        checksums[name] = digest(directory / name)
    return np.stack(arrays, axis=2), checksums






def evaluate_masks(cfg, scope="validation"):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    require_native_images(cfg)
    verify_dataset(cfg)
    if scope not in ("validation", "development_oof"):
        raise ValueError("scope只能是validation或development_oof。")
    loc, cohort = locations(cfg), read_cohort(cfg)
    cases = cohort.query("split == 'validation'" if scope == "validation" else "split == 'development'")
    rows, prediction_hashes = [], {}
    for _, patient in cases.iterrows():
        _, truth, _ = selected_patient(cfg, patient)
        prediction, hashes = predicted_slices(cfg, patient, scope, truth.shape[:2])
        prediction_hashes.update(hashes)
        for label in (1, 2, 3):
            a, b = truth == label, prediction == label
            na, nb = int(a.sum()), int(b.sum())
            distances, undefined = [], False
            for index in range(4):
                ai, bi = a[:, :, index], b[:, :, index]
                if not ai.any() and not bi.any():
                    continue
                if not ai.any() or not bi.any():
                    undefined = True
                    continue
                sa, sb = ai ^ binary_erosion(ai, border_value=0), bi ^ binary_erosion(bi, border_value=0)
                distances.extend([distance_transform_edt(~sb)[sa], distance_transform_edt(~sa)[sb]])
            rows.append({"mr": patient.mr, "scope": scope, "label": label, "true_pixels": na, "predicted_pixels": nb,
                "dice": float(2 * (a & b).sum() / (na + nb)) if na + nb else np.nan,
                "hd95_pixels": float(np.percentile(np.concatenate(distances), 95)) if distances and not undefined else np.nan,
                "status": "both_empty" if not na + nb else "empty_slice_hd95_undefined" if undefined else "ok"})
    table = pd.DataFrame(rows)
    table.to_csv(loc["root"] / f"{scope}_segmentation_metrics.csv", index=False)
    summary = table.groupby("label").agg(n_cases=("mr", "size"), n_dice_defined=("dice", "count"), mean_dice=("dice", "mean"),
        n_hd95_defined=("hd95_pixels", "count"), mean_hd95_pixels=("hd95_pixels", "mean"))
    summary.to_csv(loc["root"] / f"{scope}_segmentation_summary.csv")
    dump_json(loc["root"] / f"{scope}_evaluation_provenance.json", {
        "cohort_identity": cohort_identity(cfg), "scope": scope, "predictions_sha256": prediction_hashes,
        "artifacts": {name: digest(loc["root"] / name) for name in
            (f"{scope}_segmentation_metrics.csv", f"{scope}_segmentation_summary.csv")}})
    return table, summary






def build_validation_mask_cache(cfg):
    from scipy.ndimage import zoom
    require_native_images(cfg)
    verify_dataset(cfg)
    loc = locations(cfg)
    target = loc["root"] / "validation_masks.npz"
    fresh(target)
    cases = read_cohort(cfg).query("split == 'validation'").reset_index(drop=True)
    result = np.zeros((len(cases), 4, 3, 224, 224), dtype=np.uint8)
    hashes = {}
    for i, patient in cases.iterrows():
        _, truth, _ = selected_patient(cfg, patient)
        labels, checksums = predicted_slices(cfg, patient, "validation", truth.shape[:2])
        hashes.update(checksums)
        for index in range(4):
            for label in (1, 2, 3):
                result[i, index, label - 1] = zoom((labels[:, :, index] == label).astype(np.uint8),
                    (224 / labels.shape[0], 224 / labels.shape[1]), order=0, prefilter=False)
    np.savez_compressed(target, mrs=cases.mr.to_numpy(dtype=str), masks=result)
    dump_json(loc["root"] / "automatic_input_note.json", {"cohort_identity": cohort_identity(cfg),
        "scope": "validation_only", "experiment": "manual_model_auto_mask_substitution",
        "manual_slice_selection_still_required": True, "protocol": "selected_four_slices_2d_pixels_v1",
        "mask_cache_sha256": digest(target), "predictions_sha256": hashes,
        "note": "只替换已训练分类器的病灶输入；不代表自动取层或自动掩模训练的联合模型。"})
    return target






def verify_validation_mask_cache(cfg):
    loc = locations(cfg)
    if (loc["root"] / "transfer_manifest.json").exists():
        verify_transfer(cfg, loc["root"], "segmentation_results")
    metadata = json.loads((loc["root"] / "automatic_input_note.json").read_text(encoding="utf-8"))
    if metadata["cohort_identity"] != cohort_identity(cfg) or metadata["protocol"] != "selected_four_slices_2d_pixels_v1":
        raise ValueError("自动掩模与当前患者或取层协议不一致。")
    if digest(loc["root"] / "validation_masks.npz") != metadata["mask_cache_sha256"]:
        raise ValueError("自动分类掩模缓存发生变化。")
    cases = read_cohort(cfg).query("split == 'validation'")
    with np.load(loc["root"] / "validation_masks.npz", allow_pickle=False) as cache:
        if cache["mrs"].astype(str).tolist() != cases.mr.tolist() or cache["masks"].shape != (len(cases), 4, 3, 224, 224):
            raise ValueError("自动掩模患者顺序或形状不一致。")
        if not set(np.unique(cache["masks"])) <= {0, 1} or np.any(cache["masks"].sum(axis=2) > 1):
            raise ValueError("自动掩模必须为互斥的三通道二值标签。")
    if not (loc["root"] / "transfer_manifest.json").exists():
        for name, checksum in metadata["predictions_sha256"].items():
            if digest(loc["validation"] / name) != checksum:
                raise ValueError(f"原预测发生变化：{name}")






def export_segmentation_results(cfg, destination):
    verify_validation_mask_cache(cfg)
    for scope in ("validation", "development_oof"):
        verify_segmentation_evaluation(cfg, scope)
    files = ["validation_masks.npz", "automatic_input_note.json", "dataset_provenance.json", "case_index.csv", "splits_final.json",
        "validation_segmentation_metrics.csv", "validation_segmentation_summary.csv",
        "development_oof_segmentation_metrics.csv", "development_oof_segmentation_summary.csv",
        "validation_evaluation_provenance.json", "development_oof_evaluation_provenance.json"]
    return export_transfer(cfg, locations(cfg)["root"], files, destination, "segmentation_results", "selected_four_slices_2d_pixels_v1")






def export_segmentation_weights(cfg, destination):
    from importlib.metadata import version
    loc = locations(cfg)
    verify_dataset(cfg)
    model = loc["results"] / loc["name"] / "nnUNetTrainer__nnUNetPlans__2d"
    files = [model / "dataset.json", model / "plans.json"]
    if (model / "dataset_fingerprint.json").is_file():
        files.append(model / "dataset_fingerprint.json")
    for fold in range(5):
        folder = model / f"fold_{fold}"
        candidates = [folder / name for name in ("checkpoint_final.pth", "checkpoint_final.safetensors") if (folder / name).is_file()]
        if len(candidates) != 1:
            raise ValueError(f"fold_{fold} 需要唯一的最终权重，当前找到{len(candidates)}个。")
        files += candidates
        files += [file for file in folder.iterdir() if file.suffix in (".json", ".pkl")]
    environment = loc["root"] / "model_environment.json"
    dump_json(environment, {"nnunetv2": version("nnunetv2"), "torch": version("torch"),
        "model_relative_path": model.relative_to(loc["root"]).as_posix(),
        "input": "native-resolution selected slices exported by this notebook; not the 224x224 classifier cache"})
    files += [environment, loc["root"] / "dataset_provenance.json", loc["root"] / "splits_final.json"]
    return export_transfer(cfg, loc["root"], [file.relative_to(loc["root"]).as_posix() for file in files],
                           destination, "segmentation_weights", "selected_four_slices_2d_pixels_v1")






def imported_segmentation_inference_plan(cfg, input_folder, output_folder):
    root = paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    verify_transfer(cfg, root, "segmentation_weights")
    meta = json.loads((root / "model_environment.json").read_text(encoding="utf-8"))
    from importlib.metadata import version
    if version("nnunetv2") != meta["nnunetv2"]:
        raise ValueError("恢复推理时请使用导出记录中的 nnunetv2 版本：" + meta["nnunetv2"])
    return ["nnUNetv2_predict_from_modelfolder", "-i", str(input_folder), "-o", str(output_folder),
            "-m", str(root / meta["model_relative_path"]), "-f", "0", "1", "2", "3", "4", "-device", cfg["segmentation"]["device"]]






def verify_segmentation_evaluation(cfg, scope):
    loc = locations(cfg)
    meta = json.loads((loc["root"] / f"{scope}_evaluation_provenance.json").read_text(encoding="utf-8"))
    if meta["cohort_identity"] != cohort_identity(cfg) or meta["scope"] != scope:
        raise ValueError("分割评价与当前患者或评价范围不同。")
    for name, checksum in meta["artifacts"].items():
        if digest(loc["root"] / name) != checksum:
            raise ValueError(f"分割指标发生变化：{name}")
    cases = read_cohort(cfg).set_index("mr")
    for name, checksum in meta["predictions_sha256"].items():
        mr = name.split("_s", 1)[0]
        directory = loc["validation"] if scope == "validation" else loc["oof"] / f"fold_{int(cases.loc[mr, 'development_fold']) - 1}"
        if digest(directory / name) != checksum:
            raise ValueError(f"{name}在评价后发生变化，请重新评价。")




## 1. 导出四层数据集

在数据电脑执行。确认 `patients_root` 与现有患者清单对应；这一步不需要上传原始20多GB数据。


In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    display(export_dataset(cfg))


## 2. 查看本机将执行的命令

这一单元只打印命令。数据电脑需要已有可用的PyTorch、nnU-Net及影像读取环境；Notebook不自动安装依赖。GPU型号确认后再按显存调整nnU-Net规划，先使用2d默认规划。


In [ ]:
for phase, commands in command_plan(cfg).items():
    print(phase)
    for command in commands:
        print(shlex.join(command))


## 3. 预处理与安装固定患者划分

先开预处理开关，运行完成后再开安装划分开关。训练会再次核对划分，已有不同划分时停止，不覆盖。


In [ ]:
RUN_PREPROCESS = False
if RUN_PREPROCESS:
    run_phase(cfg, "preprocess")


In [ ]:
INSTALL_SPLITS = False
if INSTALL_SPLITS:
    print(install_splits(cfg))


## 4. 训练五折模型

五个模型依次训练，每个模型均不接触自己的开发留出患者。使用固定最终权重做折外预测和时间验证。


In [ ]:
RUN_SEGMENTATION_TRAINING = False
if RUN_SEGMENTATION_TRAINING:
    run_phase(cfg, "train")


## 5. 开发折外预测与评价

每位开发患者由没有用该患者训练的那一折模型预测。先核查这一阶段的分割质量，再冻结方案进入时间验证。


In [ ]:
RUN_OOF_PREDICTION = False
if RUN_OOF_PREDICTION:
    run_phase(cfg, "predict_oof")


In [ ]:
EVALUATE_DEVELOPMENT_SEGMENTATION = False
if EVALUATE_DEVELOPMENT_SEGMENTATION:
    detail, summary = evaluate_masks(cfg, "development_oof")
    display(summary)


## 6. 时间验证预测、评价及分类掩模

五折模型集成预测时间验证患者。最近邻缩放生成与分类缓存同形状的四层、三通道掩模。06可以把它替换到原分类器输入中，观察性能变化；这只是输入替换实验，分类器仍是用人工掩模训练的。


In [ ]:
RUN_VALIDATION_MASK_PREDICTION = False
if RUN_VALIDATION_MASK_PREDICTION:
    run_phase(cfg, "predict_validation")


In [ ]:
RUN_SEGMENTATION_EVALUATION_AND_CACHE = False
if RUN_SEGMENTATION_EVALUATION_AND_CACHE:
    detail, summary = evaluate_masks(cfg, "validation")
    display(summary)
    print(build_validation_mask_cache(cfg))


## 7. 导出结果包或权重包

结果包包括四层自动掩模、开发和时间验证Dice/HD95、患者与处理记录，适合上传到只有分类缓存的算力机。权重包包含五折最终权重、plans、dataset和版本信息，供同样安装nnU-Net的环境恢复推理。两种包可分别导出。

只传权重不能从分类缓存恢复原分辨率影像。需要在异地做新预测时，还要提供按本页相同方式导出的单层三序列输入。


In [ ]:
RUN_EXPORT_SEGMENTATION_RESULTS = False
RUN_EXPORT_SEGMENTATION_WEIGHTS = False
if RUN_EXPORT_SEGMENTATION_RESULTS:
    print(export_segmentation_results(cfg, NB_ROOT / "transfer_packages" / "segmentation_results"))
if RUN_EXPORT_SEGMENTATION_WEIGHTS:
    print(export_segmentation_weights(cfg, NB_ROOT / "transfer_packages" / "segmentation_weights"))


## 8. 在算力机导入（跳过本页训练步骤）

上传整个数据包目录，运行上面的定义单元，然后执行这里。结果包导入后直接去06开启自动掩模对照。权重包导入后可以打印恢复推理命令；输入仍须是本页导出的原分辨率单层影像。


In [ ]:
RUN_IMPORT_SEGMENTATION_RESULTS = False
RUN_IMPORT_SEGMENTATION_WEIGHTS = False
if RUN_IMPORT_SEGMENTATION_RESULTS:
    print(import_transfer(cfg, NB_ROOT / "incoming" / "segmentation_results", "segmentation"))
if RUN_IMPORT_SEGMENTATION_WEIGHTS:
    print(import_transfer(cfg, NB_ROOT / "incoming" / "segmentation_weights", "segmentation_weights"))


In [ ]:
SHOW_IMPORTED_INFERENCE_COMMAND = False
if SHOW_IMPORTED_INFERENCE_COMMAND:
    command = imported_segmentation_inference_plan(cfg, NB_ROOT / "incoming" / "imagesTs", NB_ROOT / "new_predictions")
    print(shlex.join(command))
